# 01c — Train FLUX.1-dev Character LoRA (ai-toolkit)  — Iteration 2

Replaces the SDXL DreamBooth approach (01b). Fixes iteration-1 problems:
- **Realism / sharp eyes+face** → FLUX.1-dev is far more photoreal than SDXL base
- **Tighter likeness** → FLUX handles identity better; rank 32
- **Steerability** → ai-toolkit trains on your PER-IMAGE JoyCaption detailed captions
  (the diffusers DreamBooth script ignored captions and used one instance prompt)
- **ComfyUI format** → ai-toolkit outputs a ComfyUI-native LoRA (fixes the zero-effect bug)

**Runtime:** A100 (40 or 80 GB). **Prereqs:**
1. Cleaned reference images + `.txt` detailed captions in
   `Drive/ai_character_studio/characters/<NAME>/reference-images/` (image + matching .txt).
   Each caption should start with the trigger token, e.g. `sks_vyuna, a woman with ...`.
2. A HuggingFace account with **FLUX.1-dev access granted** at
   https://huggingface.co/black-forest-labs/FLUX.1-dev (click 'Agree'), and a read token.

**Design notes (lessons from iter-1):**
- ai-toolkit runs in an ISOLATED uv venv (Python 3.11) so Colab's system
  numpy 2.5 / scipy / transformers can't break it (the dep hell from 01b).
- All long-running output goes to LOG FILES, never an unread PIPE (which deadlocks).

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
TRAIN_STEPS    = 2500      # FLUX: 2000-3000 typical for a person
LORA_RANK      = 32        # 16-32 typical; 32 for more identity detail
LEARNING_RATE  = 1e-4
RESOLUTIONS    = [768, 1024]   # FLUX trains well at 1024; multi-res bucketing
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE  = '/content/drive/MyDrive/ai_character_studio'
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
LORAS_DIR   = f'{DRIVE_BASE}/loras'
OUTPUT_LORA = f'{LORAS_DIR}/{CHARACTER_NAME}_flux.safetensors'
os.makedirs(LORAS_DIR, exist_ok=True)

# Sanity: count images and matching captions
imgs = [f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
caps = [f for f in os.listdir(REF_DIR) if f.lower().endswith('.txt')]
print(f'Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'Images: {len(imgs)} | Caption .txt files: {len(caps)}')
missing = [f for f in imgs if not os.path.exists(os.path.join(REF_DIR, os.path.splitext(f)[0]+'.txt'))]
if missing:
    print(f'⚠️  {len(missing)} images have NO matching .txt caption:', missing[:5])
else:
    print('✅ Every image has a matching caption.')
# Peek one caption to confirm trigger is present
if caps:
    sample = open(os.path.join(REF_DIR, caps[0])).read()
    print(f'\nSample caption ({caps[0]}):\n  {sample[:200]}')
    if TRIGGER_TOKEN not in sample:
        print(f'⚠️  Trigger "{TRIGGER_TOKEN}" not found in this caption — captions should include it.')

## 2. HuggingFace login (FLUX.1-dev is gated)

In [ ]:
# FLUX.1-dev requires accepting the license + a token. Two ways:
# 1 (recommended): add HF_TOKEN to Colab Secrets (key icon, left sidebar)
# 2: paste when prompted below
import os
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    from getpass import getpass
    hf_token = getpass('Paste your HuggingFace token (needs FLUX.1-dev access): ')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
# Persist HF cache to Drive so the 24GB FLUX download survives sessions
os.environ['HF_HOME'] = '/content/hf_cache'   # LOCAL disk — a Drive-backed HF cache corrupts large gated models (FUSE atomic-rename); re-download each session
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Verify access to the gated repo
!pip install -q huggingface_hub
from huggingface_hub import HfApi
try:
    HfApi().model_info('black-forest-labs/FLUX.1-dev', token=hf_token)
    print('✅ FLUX.1-dev access confirmed.')
except Exception as e:
    print('❌ No access to FLUX.1-dev. Accept the license at')
    print('   https://huggingface.co/black-forest-labs/FLUX.1-dev  then rerun.')
    print('   Error:', str(e)[:200])

## 3. Install ai-toolkit in an isolated uv venv
Keeps it away from Colab's Python 3.13 / numpy 2.5 / scipy that broke ai-toolkit in 01b.

In [ ]:
import subprocess, os
TOOLKIT_DIR = '/content/ai-toolkit'
VENV = f'{TOOLKIT_DIR}/.venv'
PY = f'{VENV}/bin/python'

# uv = fast installer + can provision a clean Python 3.11 (isolated from Colab 3.13)
!pip install -q uv

if not os.path.exists(TOOLKIT_DIR):
    !git clone https://github.com/ostris/ai-toolkit.git {TOOLKIT_DIR}
    !cd {TOOLKIT_DIR} && git submodule update --init --recursive

# Create an isolated venv with its own Python 3.11 (uv downloads it)
!uv venv --python 3.11 {VENV}

# ── Install order is LOAD-BEARING. Do not reorder. ─────────────────────────
# ai-toolkit does NOT pin torch or numpy, and pulls diffusers from a git commit.
# Getting this wrong = the ABI / infer_schema crashes we hit in earlier runs:
#
#   1) numpy 1.26 FIRST — before any torch / C-ext build. If a package is built
#      against numpy 2.x and then runs under numpy 1.x, ai-toolkit's C-ext raises
#      "numpy.dtype size changed" (an ABI crash). Pin it early.
#
#   2) requirements.txt — pulls the git-pinned diffusers + transformers + peft +
#      torch the trainer expects. torch is deliberately NOT pinned here: it
#      resolves to the current release (>=2.7), which is exactly what fixes the
#      original crash. ai-toolkit's NVFP4 op (convrot_quant.py) is typed with a
#      PEP585 builtin `list[torch.Tensor]` return; torch 2.6's infer_schema only
#      understood the old typing.List[...] form and raised
#      "Return has unsupported type list[torch.Tensor]". torch >=2.7 normalizes
#      builtin generics so the custom_op registers cleanly. (The old hard pin
#      torch==2.7.1 is gone — that exact build is no longer on the index and made
#      uv abort resolution.)
#
#   3) torchaudio — imported by toolkit/config_modules.py but NOT in
#      requirements.txt, so install it explicitly. No pin: uv picks the build that
#      matches the torch requirements just installed.
#
#   4) Re-assert numpy 1.26 with --no-deps in case requirements nudged it.
!uv pip install --python {PY} "numpy==1.26.4"
!uv pip install --python {PY} -r {TOOLKIT_DIR}/requirements.txt
!uv pip install --python {PY} huggingface_hub hf_transfer
!uv pip install --python {PY} torchaudio
!uv pip install --python {PY} --no-deps "numpy==1.26.4"

# Verify: the exact things that have broken before must all import clean —
# torch>=2.7 (infer_schema), numpy 1.26 (ABI), the NVFP4 custom_op from the
# original traceback, config_modules (pulls torchaudio), AND a real bf16 CUDA
# matmul. If any of these fail we want it HERE, not deep in the train cell.
chk = subprocess.run([PY, '-c',
    'import numpy, torch, torchaudio, diffusers, transformers, safetensors; '
    'from diffusers.schedulers.scheduling_dpmsolver_multistep import DPMSolverMultistepScheduler; '
    'import sys; sys.path.insert(0, "/content/ai-toolkit"); '
    'from toolkit.util.convrot_quant import quantize_nvfp4; '      # the module from the original traceback
    'from toolkit.config_modules import DatasetConfig; '          # imports torchaudio at module load
    'assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 7), "need torch>=2.7"; '
    'assert tuple(map(int, numpy.__version__.split(".")[:2])) == (1, 26), "need numpy 1.26"; '
    'a = torch.randn(64, 64, device="cuda", dtype=torch.bfloat16); _ = (a @ a).sum().item(); '
    'print("numpy", numpy.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available()); '
    'print("diffusers", diffusers.__version__, "transformers", transformers.__version__); '
    'print("nvfp4 custom_op + config_modules clean; bf16 CUDA matmul OK")'],
    capture_output=True, text=True)
print(chk.stdout)
if chk.returncode != 0:
    print('IMPORT/VERIFY FAILED:'); print(chk.stderr[-1500:])
else:
    print('✅ ai-toolkit venv healthy (torch>=2.7, numpy 1.26, NVFP4 op + config_modules + bf16 CUDA all clean).')

## 4. Build the FLUX LoRA training config (uses your detailed captions)

In [ ]:
import yaml

# On A100 80GB set quantize False (faster, best quality). On 40GB keep True.
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
QUANTIZE = vram_gb < 60
print(f'GPU VRAM ~{vram_gb:.0f} GB → quantize={QUANTIZE}')

config = {
  'job': 'extension',
  'config': {
    'name': f'{CHARACTER_NAME}_flux',
    'process': [{
      'type': 'sd_trainer',
      'training_folder': '/content/training_output',
      'device': 'cuda:0',
      'trigger_word': TRIGGER_TOKEN,
      'network': {'type': 'lora', 'linear': LORA_RANK, 'linear_alpha': LORA_RANK},
      'save': {'dtype': 'float16', 'save_every': 500, 'max_step_saves_to_keep': 4,
               'push_to_hub': False},
      'datasets': [{
        'folder_path': REF_DIR,
        'caption_ext': 'txt',              # ← reads your per-image detailed captions
        'caption_dropout_rate': 0.05,
        'shuffle_tokens': False,
        'cache_latents_to_disk': True,
        'resolution': RESOLUTIONS,
      }],
      'train': {
        'batch_size': 1,
        'steps': TRAIN_STEPS,
        'gradient_accumulation_steps': 1,
        'train_unet': True,
        'train_text_encoder': False,       # FLUX: text encoders stay frozen
        'gradient_checkpointing': True,
        'noise_scheduler': 'flowmatch',    # FLUX uses flow matching
        'optimizer': 'adamw8bit',
        'lr': LEARNING_RATE,
        'dtype': 'bf16',
      },
      'model': {
        'name_or_path': 'black-forest-labs/FLUX.1-dev',
        'is_flux': True,
        'quantize': QUANTIZE,              # 8-bit to fit 40GB; off on 80GB
      },
      'sample': {
        'sampler': 'flowmatch',
        'sample_every': 250,
        'width': 1024, 'height': 1024,
        'prompts': [
          f'{TRIGGER_TOKEN}, portrait photo, detailed face, sharp eyes, natural skin texture, soft window light',
          f'{TRIGGER_TOKEN}, full body, standing on a city street, candid, golden hour',
          f'{TRIGGER_TOKEN}, close-up, smiling, cinematic lighting, shallow depth of field',
        ],
        'neg': '',                          # FLUX ignores negative prompts
        'seed': 42, 'walk_seed': True,
        'guidance_scale': 4,                # FLUX distilled guidance
        'sample_steps': 20,
      },
    }],
    'meta': {'name': '[name]', 'version': '1.0'},
  }
}

CONFIG_PATH = f'/content/{CHARACTER_NAME}_flux_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print('Config written:', CONFIG_PATH)
print(f'  base=FLUX.1-dev | steps={TRAIN_STEPS} | rank={LORA_RANK} | res={RESOLUTIONS} | quantize={QUANTIZE}')

## 5. Train (runs in the isolated venv; logs to file, not PIPE)

In [ ]:
import subprocess, time, os, threading
os.makedirs('/content/training_output', exist_ok=True)
LOG = '/content/flux_train.log'

# Pass HF token + cache through to the venv process
env = {**os.environ, 'HF_HUB_ENABLE_HF_TRANSFER': '1'}

print('Starting FLUX LoRA training... first run downloads FLUX.1-dev (~24GB) to Drive HF cache.')
print(f'Live log: {LOG}\n')
start = time.time()
with open(LOG, 'w') as logf:
    proc = subprocess.Popen([PY, f'{TOOLKIT_DIR}/run.py', CONFIG_PATH],
                            cwd=TOOLKIT_DIR, stdout=logf, stderr=subprocess.STDOUT, env=env)

# Tail the log live in the cell while training runs (drains nothing — reads the file)
last = 0
while proc.poll() is None:
    time.sleep(15)
    txt = open(LOG).read()
    if len(txt) > last:
        # print only new tail lines to keep output manageable
        new = txt[last:]
        tail = '\n'.join(new.splitlines()[-4:])
        print(tail)
        last = len(txt)

print(f'\nTraining process exited ({proc.returncode}) in {(time.time()-start)/60:.1f} min')
print('Last log lines:')
print(subprocess.run(['tail','-n','20',LOG], capture_output=True, text=True).stdout)

## 6. Copy the trained LoRA to Drive

In [ ]:
import glob, shutil, os
cands = glob.glob(f'/content/training_output/{CHARACTER_NAME}_flux/*.safetensors')
if not cands:
    cands = glob.glob('/content/training_output/**/*.safetensors', recursive=True)
# prefer the final (highest step / latest mtime), skip optimizer files
cands = [c for c in cands if 'optimizer' not in c.lower()]
if cands:
    latest = max(cands, key=os.path.getmtime)
    shutil.copyfile(latest, OUTPUT_LORA)
    # also stage into ComfyUI loras if that dir exists this session
    comfy_loras = '/content/ComfyUI/models/loras'
    if os.path.isdir(comfy_loras):
        shutil.copyfile(latest, f'{comfy_loras}/{os.path.basename(OUTPUT_LORA)}')
    print(f'✅ LoRA saved: {OUTPUT_LORA}')
    print(f'   source: {latest}  ({os.path.getsize(OUTPUT_LORA)/1024**2:.1f} MB)')
else:
    print('ERROR: no .safetensors in training_output. Check the log above.')
    print(subprocess.run(['ls','-R','/content/training_output'], capture_output=True, text=True).stdout[:1000])

## 7. Update character metadata

In [ ]:
import json, os
meta_path = f'{CHAR_DIR}/metadata.json'
meta = {}
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
meta.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'flux-dev',
    'flux_lora_path': OUTPUT_LORA,
    'train_steps': TRAIN_STEPS,
    'lora_rank': LORA_RANK,
    'trainer': 'ai-toolkit',
})
json.dump(meta, open(meta_path,'w'), indent=2)
print('metadata.json updated:'); print(json.dumps(meta, indent=2))
print('\n✅ Done. Validate in ComfyUI with the FLUX stills workflow + ADetailer face pass.')
print('   FLUX inference needs: flux1-dev UNet, ae.safetensors VAE, t5xxl + clip_l text encoders (see 00 setup).')

## 8. Test the LoRA in-Colab — pose × guidance × steps grid

Loads FLUX.1-dev + your LoRA via diffusers and renders a validation grid so you can judge
likeness and pick inference settings **before** committing to a retrain.

**What this cell group does**
1. **Self-check (critical):** renders the same prompt with and without the LoRA at a fixed
   seed and reports the pixel delta. ai-toolkit saves LoRAs in its own keymap, so if the
   diff is ~0 the LoRA isn't being picked up by diffusers and needs a key conversion —
   we catch it here instead of staring at 80 un-triggered images.
2. **Pose grid:** `N_PROMPTS` distinct `sks_vyuna` poses, each rendered across a
   `guidance` × `num_steps` sweep. Images land on Drive under `lora_tests/<lora_tag>/`.

**⚠️ About "CFG" on FLUX.1-dev:** this is the *guidance-distilled* model. It has no classic
classifier-free `guidance_scale`; the analogous knob is the `guidance` **embedding**
(diffusers default 3.5, sane band ~1.5–6). The requested 8–14 sweep drives that `guidance`
value, so expect the top of the range to run hot / oversaturated. A known-good `guidance=3.5`
image is saved as the reference. To instead sweep the FLUX-native range, set
`GUIDANCES = [3.5, 4.5, 5.5]` in the config cell below and re-run.

Runs in the **same isolated venv** that did the training (matching diffusers/torch versions),
long output to a log file (never a PIPE).

In [ ]:
# ── Ensure a COMPLETE FLUX.1-dev on fast LOCAL /content storage ────────────────
# Downloaded straight to /content (local SSD, atomic-safe). NO Drive mirror:
# /content is wiped on a runtime reset, so we simply re-download here. snapshot_download
# is resumable/idempotent — it skips files already present (a re-run in the same
# session is a no-op) and writes whole files atomically, so no partial leftovers.
import os
from huggingface_hub import snapshot_download
FLUX_LOCAL = '/content/flux_dev'
os.makedirs(FLUX_LOCAL, exist_ok=True)
print('Ensuring FLUX.1-dev on local /content (downloads only what is missing)...')
snapshot_download('black-forest-labs/FLUX.1-dev',
                  local_dir=FLUX_LOCAL, local_dir_use_symlinks=False, max_workers=8)
print('FLUX_LOCAL ready:', FLUX_LOCAL)


In [ ]:
import os

# ─── Test config ───────────────────────────────────────────────────────────
LORA_FILE = OUTPUT_LORA          # the LoRA to test (rank-32 by default; point at r64 later)
LORA_TAG  = os.path.splitext(os.path.basename(LORA_FILE))[0]   # e.g. "Yuna_flux"
LORA_WEIGHT = 1.4                # validated on real gens 2026-09-06 (full likeness, saturates ~1.4).
                                 # If likeness is too weak try 1.0; too strong/artefact try 0.6.
N_PROMPTS = 6                    # how many distinct sks_vyuna poses to render
GUIDANCES = [8, 10, 12, 14]      # the "CFG" sweep you asked for (drives the FLUX guidance
                                 # embedding — see the note above about the native range)
NUM_STEPS = [16, 24, 32, 48, 64] # the steps sweep you asked for
SIZE = (1024, 1024)               # (width, height)
SEED = 42                        # shared base seed; each pose gets a different seed (SEED+i)
SELF_CHECK_SEED = 1234           # fixed seed for the with/without-LoRA effect test
SELF_CHECK_GUIDANCE = 3.5        # known-good FLUX guidance for the self-check + reference
SELF_CHECK_STEPS = 28
# ───────────────────────────────────────────────────────────────────────────

# A spread of poses/scenarios — the trigger token is prepended by the grid below.
# Edit freely; the trigger is always included.
POSES = [
    "sitting on a park bench, soft afternoon light, looking at camera",
    "standing in a rain-soaked city street, neon reflections, candid",
    "close-up portrait, head tilted slightly, window light, shallow depth of field",
    "full body in a flowing dress, turning, wind in hair, golden hour",
    "lying on a beach, looking up at the sky, overcast, film grain",
    "mid-stride on a wooden pier, backlit, silhouette rim light",
    "in a cozy cafe, holding a ceramic cup, warm interior light",
    "dancing under string lights at night, motion, bokeh",
    "leaning against a brick wall, over-the-shoulder, natural skin texture",
    "full length fashion shot, plain studio backdrop, soft even lighting",
]

TEST_ROOT = f'{DRIVE_BASE}/lora_tests'
TEST_DIR  = f'{TEST_ROOT}/{LORA_TAG}'
os.makedirs(TEST_DIR, exist_ok=True)
n_poses = min(N_PROMPTS, len(POSES))
n_cells = n_poses * len(GUIDANCES) * len(NUM_STEPS)
print(f'Testing LoRA : {LORA_FILE}')
print(f'  tag={LORA_TAG}  weight={LORA_WEIGHT}  size={SIZE}')
print(f'  poses={n_poses}  guidances={GUIDANCES}  steps={NUM_STEPS}')
print(f'  → {n_cells} images + 1 self-check pair, to {TEST_DIR}')


In [ ]:
# Runs in the same isolated venv that did the training (PY). Log to a file.
# Loads the pipeline from the complete local copy (FLUX_LOCAL), not the broken
# Drive repo id. The trigger token is prepended to every pose prompt.
import subprocess, os, time, json
LOG = '/content/flux_lora_test.log'

script = f'''
import os, json, time, numpy as np, torch
from diffusers import FluxPipeline
from safetensors.torch import load_file, save_file

LORA_PATH = {json.dumps(LORA_FILE)}
MODEL_DIR = {json.dumps(FLUX_LOCAL)}          # complete local FLUX.1-dev copy
TRIGGER   = {json.dumps(TRIGGER_TOKEN)}
LORA_WEIGHT = {LORA_WEIGHT}
POSES = {json.dumps(POSES)}
N = {int(n_poses)}
GUIDANCES = {json.dumps(GUIDANCES)}
STEPS = {json.dumps(NUM_STEPS)}
W, H = {int(SIZE[0])}, {int(SIZE[1])}
SEED = {int(SEED)}
SC_SEED, SC_GUIDE, SC_STEPS = {int(SELF_CHECK_SEED)}, {float(SELF_CHECK_GUIDANCE)}, {int(SELF_CHECK_STEPS)}
OUT = {json.dumps(TEST_DIR)}
os.makedirs(OUT, exist_ok=True)

def full_prompt(pose):
    p = f"photorealistic, {{pose}}".replace("{{pose}}", pose)
    return f"{{TRIGGER}}, {{p}}" if TRIGGER not in p else p

print("Loading FLUX.1-dev from local copy (bf16)...", flush=True)
t0 = time.time()
pipe = FluxPipeline.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16)
# FLUX.1-dev is ~40GB of weights. On a big GPU put it all on-device (fast);
# on a 40GB box offload the (frozen, used-once) text encoders to CPU.
_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
if _vram >= 60:
    pipe.to("cuda")
else:
    pipe.enable_model_cpu_offload()
torch.cuda.empty_cache()
print(f"  pipeline ready in {{time.time()-t0:.0f}}s  (VRAM {{_vram:.0f}}GB, offload={{_vram < 60}})", flush=True)

# Load the ai-toolkit LoRA state_dict, hand it to diffusers (safe_load/convert
# applies the ai-toolkit -> diffusers keymap). We print the key map so a
# mismatch is visible instead of silently producing un-triggered images.
sd = load_file(LORA_PATH)
print(f"  LoRA loaded: {{len(sd)}} tensors, e.g. {{list(sd)[:2]}}", flush=True)
tmp_lora = "/content/_lora_attach.safetensors"
save_file({{k: v.detach().cpu().contiguous() for k, v in sd.items()}}, tmp_lora)
try:
    keymap = pipe.load_lora_weights(tmp_lora)
    print("  load_lora_weights ->", keymap, flush=True)
except Exception as e:
    print("  load_lora_weights raised:", repr(e), "-> fallback load_lora_adapter", flush=True)
    pipe.load_lora_adapter(tmp_lora)

# diffusers >= 0.32: set_adapters([names], [weights]); load_lora_weights registers
# the adapter as "default_0". Older builds use set_adapters([weights]).
def set_strength(pipe, s):
    try:
        pipe.set_adapters(["default_0"], [s])
    except Exception:
        pipe.set_adapters([s])

set_strength(pipe, LORA_WEIGHT)
torch.cuda.empty_cache()

def gen(prompt, seed, guidance, steps):
    g = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=prompt, height=H, width=W, guidance_scale=guidance,
                num_inference_steps=steps, generator=g).images[0]

def np_of(pil): return np.asarray(pil.convert("RGB")).astype(np.int16)

print("SELF-CHECK: does the LoRA change the image?", flush=True)
sc_prompt = full_prompt(POSES[0])
set_strength(pipe, LORA_WEIGHT);  with_lora    = gen(sc_prompt, SC_SEED, SC_GUIDE, SC_STEPS)
set_strength(pipe, 0.0);          without_lora = gen(sc_prompt, SC_SEED, SC_GUIDE, SC_STEPS)
set_strength(pipe, LORA_WEIGHT)
d = float(np.abs(np_of(with_lora) - np_of(without_lora)).mean())
with_lora.save(f"{{OUT}}/selfcheck_with_lora.png")
without_lora.save(f"{{OUT}}/selfcheck_without_lora.png")
with_lora.save(f"{{OUT}}/ref_g{{SC_GUIDE}}_s{{SC_STEPS}}.png")   # known-good reference
print(f"  self-check mean abs pixel diff = {{d:.3f}}  (0-60 scale)", flush=True)
print("  APPLIED" if d >= 3.0 else "  WARNING: negligible diff — LoRA likely NOT applied", flush=True)

print("POSE x GUIDANCE x STEPS GRID", flush=True)
rows, t0 = [], time.time()
for i, pose in enumerate(POSES[:N]):
    fp, seed = full_prompt(pose), SEED + i
    for g in GUIDANCES:
        for s in STEPS:
            fn = f"p{{i+1:02d}}_g{{g}}_s{{s}}.png"
            gen(fp, seed, g, s).save(f"{{OUT}}/{{fn}}")
            rows.append({{"prompt": fp, "seed": seed, "guidance": g, "steps": s,
                          "weight": LORA_WEIGHT, "file": fn}})
            print(f"  p{{i+1:02d}} g{{g}} s{{s}}  ({{time.time()-t0:.0f}}s)", flush=True)
    torch.cuda.empty_cache()

json.dump({{"lora": LORA_PATH, "weight": LORA_WEIGHT, "model_dir": MODEL_DIR,
           "trigger": TRIGGER, "guidances": GUIDANCES, "steps": STEPS, "size": [W, H],
           "selfcheck_diff": d, "images": rows}},
          open(f"{{OUT}}/test_meta.json", "w"), indent=2)
print(f"\\nDone. {{len(rows)}} images + self-check pair. Self-check diff={{d:.2f}}.", flush=True)
print("Browse:", OUT, flush=True)
'''

env = {**os.environ, 'HF_HUB_ENABLE_HF_TRANSFER': '1'}
print(f"Running LoRA test → log {LOG}\n")
t0 = time.time()
with open(LOG, 'w') as logf:
    proc = subprocess.run([PY, '-c', script], cwd='/content', stdout=logf,
                          stderr=subprocess.STDOUT, env=env)
print(f"exit={proc.returncode} in {(time.time()-t0)/60:.1f} min")
print("---- tail of log ----")
print(subprocess.run(['tail','-n','30',LOG], capture_output=True, text=True).stdout)
if proc.returncode != 0:
    print("\nFull error tail:")
    print(open(LOG, errors='ignore').read()[-3000:])

## 9. Retrain at rank 64 (distinct name/config — never overwrites rank 32)

The rank-32 run's loss was **flat** (0.40 → 0.38 over 2500 steps) — it underfit, which is why
the character didn't come through. Two things changed for rank 64:

1. **EMA is now ON.** The rank-32 run trained *without* EMA, but the official ai-toolkit FLUX
   recipe turns it on (`ema_decay: 0.99`). EMA smooths the weights over many steps and is the
   single biggest lever for identity stability — leaving it off was the main reason r32 was
   weak. This is the real fix, not just "make it bigger."
2. **Learning rate: raised `1e-4 → 2e-4`** (see the reasoning note below).

Everything is named `..._r64` so `Yuna_flux.safetensors` (rank 32) is **untouched**.

**LR investigation (why 2e-4, and why not more):**
- The r32 curve was *declining, not converged* — loss was still dropping at step 2500. That
  says **undertrained**, which points *up* on LR/steps, not down. Raising LR when the loss is
  still falling is the direct fix; lowering it would make it worse.
- We did **not** go higher than 2e-4 because (a) with EMA on, effective convergence is faster,
  and (b) higher LR on a high-rank FLUX LoRA risks identity drift / over-saturation. 2e-4 is
  the aggressive-but-safe value. If the r64 loss *still* flattens above ~0.30 by step 2500,
  bump `R64_LR` to `3e-4` or extend `R64_STEPS` to 3000 — single-number edits below.
- **If you'd rather be conservative**, set `R64_LR = 1e-4` — with EMA on, even 1e-4 should
  clearly beat the (EMA-off) r32. The EMA change is doing most of the work.

In [ ]:
import yaml, torch, os, subprocess

# ─── Rank-64 config — all names carry _r64 so rank 32 is never touched ─────
R64_RANK     = 64
R64_LR       = 2e-4              # investigated: r32 was underfit (flat, still
                                 # declining) → go UP, not down. 2e-4 is the
                                 # aggressive-but-safe value (see note above).
R64_STEPS    = 2500
R64_EMA      = True              # ← the key change vs r32 (official FLUX recipe)
R64_EMA_DECAY = 0.99
R64_TAG      = f'{CHARACTER_NAME}_flux_r64'      # config name + output folder
OUTPUT_LORA_R64 = f'{LORAS_DIR}/{CHARACTER_NAME}_flux_r64.safetensors'
os.makedirs(LORAS_DIR, exist_ok=True)
assert OUTPUT_LORA_R64 != OUTPUT_LORA, "r64 output must not collide with r32!"

# ── Ensure a COMPLETE FLUX.1-dev on fast LOCAL /content storage ────────────────
# The trainer MUST load from a flat local copy, never the repo id (a repo id makes
# ai-toolkit download into the HF cache, which we keep on local disk). NO Drive mirror:
# /content is wiped on a runtime reset, so we re-download here (snapshot_download is
# resumable — skips files already present, atomic writes). Duplicated from the Section 8
# loader so the rank-64 path is self-sufficient after a reset.
from huggingface_hub import snapshot_download
FLUX_LOCAL = '/content/flux_dev'
os.makedirs(FLUX_LOCAL, exist_ok=True)
print('Ensuring FLUX.1-dev on local /content (downloads only what is missing)...')
snapshot_download('black-forest-labs/FLUX.1-dev',
                  local_dir=FLUX_LOCAL, local_dir_use_symlinks=False, max_workers=8)
FLUX_SRC = FLUX_LOCAL
print(f'  FLUX source: {FLUX_SRC}')


vram_gb = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
QUANTIZE = vram_gb < 60
print(f'GPU VRAM ~{vram_gb:.0f} GB → quantize={QUANTIZE}')

config = {
  'job': 'extension',
  'config': {
    'name': R64_TAG,                                   # ← distinct folder: Yuna_flux_r64
    'process': [{
      'type': 'sd_trainer',
      'training_folder': '/content/training_output',
      'device': 'cuda:0',
      'trigger_word': TRIGGER_TOKEN,
      'network': {'type': 'lora', 'linear': R64_RANK, 'linear_alpha': R64_RANK},
      'save': {'dtype': 'float16', 'save_every': 500, 'max_step_saves_to_keep': 4,
               'push_to_hub': False},
      'datasets': [{
        'folder_path': REF_DIR,
        'caption_ext': 'txt',
        'caption_dropout_rate': 0.05,
        'shuffle_tokens': False,
        'cache_latents_to_disk': True,
        'resolution': RESOLUTIONS,
      }],
      'train': {
        'batch_size': 1,
        'steps': R64_STEPS,
        'gradient_accumulation_steps': 1,
        'train_unet': True,
        'train_text_encoder': False,
        'gradient_checkpointing': True,
        'noise_scheduler': 'flowmatch',
        'optimizer': 'adamw8bit',
        'lr': R64_LR,
        'dtype': 'bf16',
        # ← NEW vs r32: EMA (nests under train, per official FLUX example)
        'ema_config': {'use_ema': R64_EMA, 'ema_decay': R64_EMA_DECAY},
      },
      'model': {
        'name_or_path': FLUX_SRC,
        'is_flux': True,
        'quantize': QUANTIZE,
      },
      'sample': {
        'sampler': 'flowmatch',
        'sample_every': 250,
        'width': 1024, 'height': 1024,
        'prompts': [
          f'{TRIGGER_TOKEN}, portrait photo, detailed face, sharp eyes, natural skin texture, soft window light',
          f'{TRIGGER_TOKEN}, full body, standing on a city street, candid, golden hour',
          f'{TRIGGER_TOKEN}, close-up, smiling, cinematic lighting, shallow depth of field',
        ],
        'neg': '',
        'seed': 42, 'walk_seed': True,
        'guidance_scale': 4,
        'sample_steps': 20,
      },
    }],
    'meta': {'name': '[name]', 'version': '1.0'},
  }
}

CONFIG_PATH_R64 = f'/content/{R64_TAG}_config.yaml'
with open(CONFIG_PATH_R64, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print('Config written:', CONFIG_PATH_R64)
print(f'  base={FLUX_SRC} | steps={R64_STEPS} | rank={R64_RANK} | lr={R64_LR} '
      f'| ema={R64_EMA} ({R64_EMA_DECAY}) | res={RESOLUTIONS} | quantize={QUANTIZE}')
print(f'  output LoRA → {OUTPUT_LORA_R64}')

## 10. Train rank 64 (logs to its own file)

In [ ]:
import subprocess, time, os
os.makedirs('/content/training_output', exist_ok=True)
LOG_R64 = '/content/flux_train_r64.log'

env = {**os.environ, 'HF_HUB_ENABLE_HF_TRANSFER': '1'}
print(f'Starting RANK-64 FLUX LoRA training → {LOG_R64}\n')
start = time.time()
with open(LOG_R64, 'w') as logf:
    proc = subprocess.Popen([PY, f'{TOOLKIT_DIR}/run.py', CONFIG_PATH_R64],
                            cwd=TOOLKIT_DIR, stdout=logf, stderr=subprocess.STDOUT, env=env)
last = 0
while proc.poll() is None:
    time.sleep(15)
    txt = open(LOG_R64).read()
    if len(txt) > last:
        print('\n'.join(txt[last:].splitlines()[-4:]))
        last = len(txt)
print(f'\nRank-64 training exited ({proc.returncode}) in {(time.time()-start)/60:.1f} min')
print('Last log lines:')
print(subprocess.run(['tail','-n','20',LOG_R64], capture_output=True, text=True).stdout)


## 11. Copy the rank-64 LoRA to Drive (separate file — rank 32 kept)

In [ ]:
import glob, shutil, os
# Rank-64 output lives in its own folder (config name = Yuna_flux_r64), so this
# glob can ONLY see r64 files — the r32 LoRA is structurally safe.
cands = [c for c in glob.glob(f'/content/training_output/{R64_TAG}/*.safetensors')
         if 'optimizer' not in c.lower()]
if cands:
    latest = max(cands, key=os.path.getmtime)
    shutil.copyfile(latest, OUTPUT_LORA_R64)
    comfy_loras = '/content/ComfyUI/models/loras'
    if os.path.isdir(comfy_loras):
        shutil.copyfile(latest, f'{comfy_loras}/{os.path.basename(OUTPUT_LORA_R64)}')
    print(f'✅ Rank-64 LoRA saved: {OUTPUT_LORA_R64}')
    print(f'   source: {latest}  ({os.path.getsize(OUTPUT_LORA_R64)/1024**2:.1f} MB)')
    print(f'   (rank-32 LoRA untouched at {OUTPUT_LORA})')
else:
    print(f'ERROR: no .safetensors for {R64_TAG}. Check the log above.')
    print(subprocess.run(['ls','-R','/content/training_output'], capture_output=True, text=True).stdout[:1000])


## 12. Update character metadata (records BOTH LoRAs — r32 preserved)

In [ ]:
import json, os
meta_path = f'{CHAR_DIR}/metadata.json'
meta = {}
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
meta.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'flux-dev',
    'trainer': 'ai-toolkit',
    # Both LoRAs live side by side; 'active' points at the one to use first.
    'flux_lora': {
        'rank32': {
            'path': OUTPUT_LORA,
            'rank': 32, 'steps': TRAIN_STEPS, 'lr': LEARNING_RATE, 'ema': False,
            'note': 'first pass — flat loss (underfit), use as baseline',
        },
        'rank64': {
            'path': OUTPUT_LORA_R64,
            'rank': R64_RANK, 'steps': R64_STEPS, 'lr': R64_LR,
            'ema': R64_EMA, 'ema_decay': R64_EMA_DECAY,
            'note': 'EMA on + raised LR — should capture identity better',
        },
    },
    # keep the original flat keys pointing at the newer (rank-64) LoRA
    'flux_lora_path': OUTPUT_LORA_R64,
    'lora_rank': R64_RANK,
    'train_steps': R64_STEPS,
})
json.dump(meta, open(meta_path,'w'), indent=2)
print('metadata.json updated (both LoRAs recorded):')
print(json.dumps(meta, indent=2))
print('\n✅ Done. Test rank-64 in the Section 8 grid by setting LORA_FILE = OUTPUT_LORA_R64.')
